# 05 - Model Training

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** 5 regression models train karna aur unhe `models/` folder me `.pkl` files me save karna.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries aur train/test data load karna
Notebook 04 ke saved splits load karte hain.

In [ ]:
import os, time, pickle
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

X_train = pd.read_csv('../datasets/processed/X_train.csv')
y_train = pd.read_csv('../datasets/processed/y_train.csv').squeeze()
print(X_train.shape, y_train.shape)

## 2. Models define karna
5 models, wahi settings jo original PyCharm project (`utils.py`) me thi:

| Model | Type |
|---|---|
| Linear Regression | simple baseline |
| Decision Tree | single tree (depth 15) |
| Random Forest | 500 trees, depth 25 (dashboard isi se predict karta hai) |
| Extra Trees | 400 trees, no depth limit |
| Gradient Boosting | 300 stages, huber loss |

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=15, min_samples_split=10, min_samples_leaf=3, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=500, max_depth=25, min_samples_split=4, min_samples_leaf=2, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=400, max_depth=None, min_samples_split=3, min_samples_leaf=1, max_features=0.9, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, min_samples_split=5, min_samples_leaf=2, loss='huber', random_state=42),
}
list(models)

## 3. Models ko train karna
Har model `fit()` hota hai aur uska training time naapte hain. Random Forest, Extra Trees aur Gradient Boosting me kuch minute lag sakte hain.

In [ ]:
times = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    times[name] = round(time.time() - t0, 1)
    print(f'{name:20s} trained in {times[name]} s')

## 4. Models ko `.pkl` me save karna
Har trained model `models/` folder me `pickle` se save hota hai. Dashboard `random_forest.pkl` load karta hai.

In [ ]:
os.makedirs('../models', exist_ok=True)
files = {
    'Linear Regression': 'linear_regression.pkl',
    'Decision Tree': 'decision_tree.pkl',
    'Random Forest': 'random_forest.pkl',
    'Extra Trees': 'extra_trees.pkl',
    'Gradient Boosting': 'gradient_boosting.pkl',
}
for name, fname in files.items():
    with open(os.path.join('../models', fname), 'wb') as f:
        pickle.dump(models[name], f)
print('Saved all models in ../models')

## 5. Saved files ka size check
Tree-based models (khaaskar Random Forest aur Extra Trees) bahut bade ho jaate hain kyunki har tree data ka bada hissa yaad rakhta hai.

In [ ]:
pd.DataFrame([
    {'file': f, 'size_MB': round(os.path.getsize(os.path.join('../models', f)) / 1e6, 1)}
    for f in sorted(os.listdir('../models')) if f.endswith('.pkl') or f.endswith('.json')
])

## 6. Summary

- 5 models train hue aur `models/` me `.pkl` ke roop me save hue (saath me `encoders.pkl` notebook 03 se aur `feature_columns.json` notebook 04 se).
- Random Forest (~850 MB) aur Extra Trees (~1.7 GB) badi files hain; Extra Trees ko dashboard load nahi karta.
- Models ka performance notebook 06 me naapenge.